In [1]:
!pip install datasets tiktoken -q

In [ ]:
import os, math, time, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

batch_size         = 32
accumulation_steps = 4
block_size         = 256
max_iters          = 10000
eval_interval      = 500
eval_iters         = 30
eval_batch_size    = 16
learning_rate      = 3e-4
min_lr             = 3e-5
warmup_iters       = 2000
n_embd             = 512
n_head             = 8
n_layer            = 8
dropout            = 0.1
vocab_size         = 50257
checkpoint_dir     = "/kaggle/working/checkpoints"
BASE_CHECKPOINT    = "/kaggle/input/models/renganathc/model/pytorch/default/1/ckpt_0026000.pt"

In [ ]:
import os
import numpy as np
import tiktoken

TXT_FILE   = "/kaggle/input/datasets/renganathc/presedential-new/us_presidential.txt"
TRAIN_BIN  = "/kaggle/working/train.bin"
VAL_BIN    = "/kaggle/working/val.bin"

enc = tiktoken.get_encoding("gpt2")

if os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN):
    print("Binary files already exist, skipping tokenization.")
else:
    print("Loading text file...")
    with open(TXT_FILE, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"Total characters in text: {len(text):,}")

    tokens = enc.encode(text)
    print(f"Total tokens: {len(tokens):,}")

    split_idx = int(len(tokens) * 0.95)
    train_tokens = tokens[:split_idx]
    val_tokens   = tokens[split_idx:]

    # Save bin
    np.array(train_tokens, dtype=np.uint16).tofile(TRAIN_BIN)
    np.array(val_tokens, dtype=np.uint16).tofile(VAL_BIN)
    print(f"Saved train.bin ({len(train_tokens):,} tokens) and val.bin ({len(val_tokens):,} tokens)")

Loading text file...
Total characters in text: 238,831
Total tokens: 41,569
Saved train.bin (39,490 tokens) and val.bin (2,079 tokens)


In [ ]:

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")
enc = tiktoken.get_encoding("gpt2")

period_ids = set(enc.encode("."))
q_ids      = set(enc.encode("?"))
e_ids      = set(enc.encode("!"))
SENTENCE_END_TOKENS = period_ids | q_ids | e_ids

def clean_start_index(data, i):
    while i < len(data) - 1:
        tok = int(data[i])
        if tok in SENTENCE_END_TOKENS:
            i += 1
        else:
            break
    return i

def find_sentence_starts(data):
    data_np = np.array(data, dtype=np.int32)
    mask = np.isin(data_np[:-1], list(SENTENCE_END_TOKENS))
    starts = np.where(mask)[0] + 1
    return starts.astype(np.int64)

sentence_starts_train = find_sentence_starts(train_data)

def get_batch(split, bs=None):
    if bs is None:
        bs = batch_size
    data = train_data if split=="train" else val_data
    use_sentence = (split=="train")
    n_sentence = bs // 2 if use_sentence else 0
    n_random   = bs - n_sentence
    ix_list = []

    if use_sentence:
        sent_ix = torch.randint(0, len(sentence_starts_train), (n_sentence * 2,))
        sent_ix = torch.from_numpy(sentence_starts_train)[sent_ix]
        sent_ix = torch.tensor([clean_start_index(data, int(i)) for i in sent_ix])
        sent_ix = sent_ix[sent_ix < len(data)-block_size][:n_sentence]
        ix_list.append(sent_ix)

    rand_ix = torch.randint(0, len(data)-block_size, (n_random,))
    ix_list.append(rand_ix)
    ix = torch.cat(ix_list, dim=0)
    ix = ix[torch.randperm(len(ix))]

    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn     = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj     = nn.Linear(n_embd, n_embd)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer(
            "bias", torch.tril(torch.ones(block_size, block_size))
                    .view(1, 1, block_size, block_size)
        )
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(n_embd, dim=2)
        k = k.view(B, T, n_head, C // n_head).transpose(1, 2)
        q = q.view(B, T, n_head, C // n_head).transpose(1, 2)
        v = v.view(B, T, n_head, C // n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.drop   = nn.Dropout(dropout)
        self.act    = nn.GELU()
    def forward(self, x):
        return self.drop(self.c_proj(self.act(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.mlp  = MLP()
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(
            self.token_embedding(idx) +
            self.position_embedding(torch.arange(T, device=device))
        )
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            targets.view(-1)
        ) if targets is not None else None
        return logits, loss

import numpy

torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])

raw_model = GPT().to(device)
ckpt = torch.load(BASE_CHECKPOINT, map_location=device, weights_only=False)
raw_model.load_state_dict(ckpt["model"])
print("Base checkpoint loaded successfully.")

In [ ]:
class LoRALinear(nn.Module):
    """LoRA wrapper for a Linear layer (device-safe)."""
    def __init__(self, linear_module, r=12, alpha=16, dropout=0.1):
        super().__init__()
        self.linear = linear_module  #      original linear
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)

        # lora matrices
        in_features = self.linear.in_features
        out_features = self.linear.out_features

        self.A = nn.Parameter(torch.zeros(in_features, r))
        self.B = nn.Parameter(torch.zeros(r, out_features))

        # initialize
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

        # freeze original weights
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

    def forward(self, x):
        A = self.A
        B = self.B
        if A.device != x.device:
            A = A.to(x.device)
        if B.device != x.device:
            B = B.to(x.device)

        return self.linear(x) + self.dropout(x @ A @ B) * self.scaling

In [ ]:

scaler = torch.cuda.amp.GradScaler()

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > max_iters:
        return min_lr
    coeff = 0.5 * (1 + math.cos(math.pi*(it - warmup_iters) / (max_iters - warmup_iters)))
    return min_lr + coeff * (learning_rate - min_lr)

def save_checkpoint(iter_num, best_val_loss):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f"lora_ckpt_{iter_num:07d}.pt")
    torch.save({
        "iter_num": iter_num,
        "best_val_loss": best_val_loss,
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict()
    }, path)
    
    print(f"  >> saved {path}")

@torch.no_grad()
def estimate_loss():
    raw_model.eval()
    torch.cuda.empty_cache()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split, bs=eval_batch_size)
            with torch.amp.autocast("cuda"):
                _, loss = raw_model(x, y)
            losses.append(loss.item())
        out[split] = np.mean(losses)
    raw_model.train()
    return out

In [ ]:
for param in raw_model.parameters():
    param.requires_grad = False

for block in raw_model.blocks:
    block.attn.c_attn = LoRALinear(block.attn.c_attn, r=12, alpha=16, dropout=0.1).to(device)
    block.attn.c_proj = LoRALinear(block.attn.c_proj, r=12, alpha=16, dropout=0.1).to(device)

trainable = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in raw_model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, raw_model.parameters()),
    lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1
)

Trainable: 294,912 / 51,377,664 (0.57%)


In [ ]:
iter_num, best_val_loss = 0, float("inf")
optimizer.zero_grad(set_to_none=True)
t0 = time.time()

while iter_num < max_iters:
    for pg in optimizer.param_groups:
        pg["lr"] = get_lr(iter_num)

    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        print(f"iter {iter_num:6d} | train {losses['train']:.4f} | val {losses['val']:.4f} | lr {get_lr(iter_num):.2e} | {(time.time()-t0)/3600:.2f}h")
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            save_checkpoint(iter_num, best_val_loss)

    for micro_step in range(accumulation_steps):
        x, y = get_batch("train")
        with torch.amp.autocast("cuda"):
            _, loss = raw_model(x, y)
            loss = loss / accumulation_steps
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    iter_num += 1

save_checkpoint(iter_num, best_val_loss)